<a href="https://colab.research.google.com/github/ParkHangah/AIFFEL_quest_eng/blob/master/LLM_Aplication/LLM04/KoChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KoChatGPT

[출력보전된 원본 코랩 파일](https://colab.research.google.com/drive/1_lBWFTOJGcEr-0iVxNgH3nJLMCHvHnKU?usp=sharing)

## 1.사전준비

#### 1)  Google 드라이브 연결

##### (1) 경로 설정

In [ ]:
# 드라이브 경로 설정  by 박항아
drive_path = '#Study/Aiffel/Work' # 평상시 작업하는 드라이브 폴더 경로를 입력해 주세요.
project_name = 'KoChatGPT'       # 이번 프로젝트세 사용하는 폴더명을 입력해주세요.

##### (2) 드라이브 연결

In [ ]:
from google.colab import drive
from IPython.display import clear_output
import os

# 1. 구글 드라이브 마운트
print("Connecting...")
drive.mount('/content/gdrive')

# 2. 경로 설정 및 폴더 생성
base_path = os.path.join('/content/gdrive/MyDrive',drive_path)
project_path = os.path.join(base_path, project_name)

# Create the project directory if it doesn't exist
os.makedirs(project_path, exist_ok=True)


# 2. 출력 전체 삭제 후 완료 메시지 출력
clear_output()
print("✅ 구글 드라이브 연결이 성공적으로 완료되었습니다.")
print(f"Selected Google Drive root path: {base_path}")

#### 2) 라이브러리 설치

In [ ]:
!pip install datasets
!pip install loralib
!pip install trl
!pip install accelerate
!pip install transformers

#### 3) 코랩용 - 라이브러리 원본 소스코드 일부 수정 (경로 수정 필요)

In [ ]:
import os

modifications = [
    {
        "file": "/content/gdrive/MyDrive/#Study/Aiffel/Work/KoChatGPT/chatgpt/trainer/callbacks/save_checkpoint.py",
        "changes": [
            {"line": 3, "old": "from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
             "new": "from chatgpt.trainer.strategies import Strategy"},
            {"line": 71, "old": "only_rank0 = not isinstance(self.strategy, ColossalAIStrategy)",
             "new": "            only_rank0 = not isinstance(self.strategy)"},
        ],
    },
    {
        "file": "/content/gdrive/MyDrive/#Study/Aiffel/Work/KoChatGPT/chatgpt/trainer/strategies/__init__.py",
        "changes": [
            {"line": 1, "old": "from .colossalai import ColossalAIStrategy", "new": ""},  # 삭제
            {"line": 5, "old": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
             "new": "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']"},
        ],
    },
    {
        "file": "/content/gdrive/MyDrive/#Study/Aiffel/Work/KoChatGPT/chatgpt/dataset/reward_dataset.py",
        "changes": [
            {"line": 3, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ],
    },
    {
        "file": "/content/gdrive/MyDrive/#Study/Aiffel/Work/KoChatGPT/chatgpt/trainer/base.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    },
    {
        "file": "/content/gdrive/MyDrive/#Study/Aiffel/Work/KoChatGPT/chatgpt/trainer/rm.py",
        "changes": [
            {"line": 8, "old": "from tqdm import tqdm", "new": "from tqdm.notebook import tqdm"},
        ]
    }
]


def modify_file(file_path, changes):
    """파일에서 지정된 줄을 찾아 내용을 수정하는 함수"""

    if not os.path.exists(file_path):
        print(f"⚠️ 파일이 존재하지 않습니다: {file_path}")
        return

    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    modified = False

    for change in changes:
        line_index = change["line"]
        if 0 <= line_index < len(lines):
            if lines[line_index].strip() == change["old"]:
                lines[line_index] = change["new"] + "\n"
                modified = True
            else:
                print(f"⚠️ {file_path} 파일의 {change['line']}번째 줄이 예상과 다릅니다.")
                print(f"   예상: {change['old']}")
                print(f"   실제: {lines[line_index].strip()}")

    if modified:
        with open(file_path, "w", encoding="utf-8") as file:
            file.writelines(lines)
        print(f"✅ 수정 완료: {file_path}")
    else:
        print(f"⚠️ {file_path} 수정할 내용이 없습니다.")

for mod in modifications:
    modify_file(mod["file"], mod["changes"])

#### 4) 소스코드 로드 및 필수 requirement설치 확인

In [ ]:
# 소스코드 설치 경로를 파이썬 검색 경로에 추가
import sys
import os

# 소스코드의 '부모 폴더' 경로를 지정
# 'chatgpt' 폴더가 들어있는 바로 상위 폴더까지의 경로
# project_path = '/content/gdrive/MyDrive/#Study/Aiffel/Work/KoChatGPT'

if project_path not in sys.path:
    sys.path.append(project_path)

# 제대로 설정되었는지 확인 (chatgpt 폴더가 목록에 보여야 함)
print(os.listdir(project_path))

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy

print("Torch version:{}".format(torch.__version__)) # Torch version:1.12.1
print("Cuda version: {}".format(torch.version.cuda)) # Cuda version: 11.3
print("transformers version: {}".format(transformers.__version__)) # transformers 4.28.0
print("GPU 사용 가능여부: {}".format(torch.cuda.is_available()))

# 만일 아래 모듈이 불러와지지 않는다면 Clone 및 수정을 잘 진행했는지 확인해주세요.
from chatgpt.trainer.strategies import NaiveStrategy

## 2.Base model and Dataset for RLHF

### 2-1. backbone 모델로 사용할 KoGPT-2의 성능 확인하기

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "skt/kogpt2-base-v2"
from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2",
                                                    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
                                                    pad_token='<pad>', mask_token='<mask>')

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

In [ ]:
# 사용할 모델의 토크나이저가 입력받아 처리할 수 있는 최대 토큰 수
tokenizer.model_max_length

In [ ]:
# kogpt-2 토크나이징 방식
model.config.n_positions

In [ ]:
input_txt = "바람도 없는 공중에 수직의 파문을 내이며 고요히 떨어지는 오동잎은 누구의 발자취 입니까."
tokens = tokenizer(input_txt).tokens()
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].numpy()
pd.options.display.max_columns = 40
pd.options.display.max_rows = 60
df = pd.DataFrame([tokens, input_ids[0]], index=["kogpt-2_tokens", "Input_IDs"])
df

In [ ]:
# 디코딩 성능도 확인
max_length=128
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids, max_length=max_length, do_sample=False)
print(tokenizer.decode(output_greedy[0]))

* 시퀀스가 반복되어 출력 (그리디 서치 디코딩시 발견되는 전형적인 현상)

In [ ]:
# 빔 서치 디코딩을 사용하고 n-gram 패널티까지 부과
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_beam = model.generate(input_ids, max_length=max_length, num_beams=10, no_repeat_ngram_size=2,
                             do_sample=False)
print(tokenizer.decode(output_beam[0]))

- 입력 시퀀스와 별 상관 없어 보이는 긴 문단이 생성
- 그럼에도 생성된 문단은 제법 맥락을 갖춤.
- 그러나 문장 간의 정합성이나 일관성은 다소 떨어지는 부분도 관찰

In [ ]:
# 샘플링 기법까지 추가
output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, temperature=2.0, top_k=50)
print(tokenizer.decode(output_beam[0]))

- `temperature`  
emperature 값을 낮추면(예: temperature=0.5) 확률이 높은 단어들이 더욱 우세하게 선택되며, 보수적인 생성이 이루어짐.
반대로 값을 높이면(예: temperature=1.5) 확률이 낮은 단어들도 더 자주 선택되며, 더 창의적이지만 불안정한 결과가 나올 수 있음
- `top_k`  
샘플링 시 상위 k개의 단어 후보만 고려하여 나머지를 무시하는 방식
예를 들어, top_k=50이면, 확률이 높은 상위 50개의 단어만 샘플링 대상이 되고, 그 외 단어는 제외

In [ ]:
# top_p 샘플링 기법 사용
output_beam = model.generate(input_ids, max_length=max_length, num_beams=7, no_repeat_ngram_size=2,
                             do_sample=True, top_p=0.90)
print(tokenizer.decode(output_beam[0]))

- `top_p`  
Nucleus Sampling이라고도 불리며, 단어 선택 시 확률 누적 기준으로 상위 p%의 단어들만 고려하는 방법

### 2-2. 데이터셋 확인

In [ ]:
import json
data_path_1_SFT =  os.path.join(project_path,"data_kochatgpt/kochatgpt_1_SFT.jsonl")
with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)

print(len(list_data_dict))
list_data_dict[:3]

In [ ]:
# RM에 사용할 데이터셋
data_path_2_RM =  os.path.join(project_path,'data_kochatgpt/kochatgpt_2_RM.jsonl')
with open(data_path_2_RM, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
print(len(list_data_dict))
list_data_dict[:3]

In [ ]:
# PPO 학습에 쓰일 데이터셋
data_path_3_PPO = os.path.join(project_path,'data_kochatgpt/kochatgpt_3_PPO.jsonl')
with open(data_path_3_PPO, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
print(len(list_data_dict))
list_data_dict[:3]

## 3.SFT (Supervised Fine-Tuning)

### 3-1. SFT - 모델구현

#### 1)라이브러리, 모델, 토크나이저 로드

In [ ]:
# 1. 필요한 라이브러리 로드
from typing import Optional, Dict, Sequence
from torch.utils.data import Dataset
from dataclasses import dataclass
import logging
import copy

In [ ]:
# 2. 모델과 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
# from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2", model_max_length=512,
                                                    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
                                                    pad_token='<pad>', mask_token='<mask>')

print(tokenizer)

#### 2)SFT 데이터셋 클래스 정의

In [ ]:
# 3. 모델 인퍼런스 단계에서 사용할 prompt 딕셔너리 템플릿과 SFT 데이터셋 클래스를 정의
class SFT_dataset(Dataset):
    def __init__(self, data_path_1_SFT: str, tokenizer: transformers.PreTrainedTokenizer, verbose=False):
        super(SFT_dataset, self).__init__()
        logging.warning("Loading data...")
        # 3-1. JSON 데이터에서 불러올 키(Key) 이름 정의
        pattern_instruction = 'prompt'  # instruction
        pattern_output = 'completion'  # response
        # 3-2. JSON 파일 로드
        with open(data_path_1_SFT, "r", encoding='utf-8-sig') as json_file:
            list_data_dict = json.load(json_file)
        # 3-3. 모델에게 보여줄 프롬프트 템플릿 설정 (입력 데이터 형식화)
        PROMPT_DICT = {
            "prompt_input": (
                "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
            )
        }
        prompt_input = PROMPT_DICT["prompt_input"]
        # 3-4. 소스(질문)와 타겟(답변) 리스트 생성
        sources = []
        for example in list_data_dict:
            # 템플릿에 맞춰 질문 생성
            # (예: ### Instruction(명령어): 안녕하세요... ### Response(응답):)
            tmp = prompt_input.format_map(example)
            sources.append(tmp)
        targets = []
        for example in list_data_dict:
            # 답변 뒤에 문장의 끝을 알리는 EOS 토큰 추가
            targets.append(f"{example[pattern_output]}{tokenizer.eos_token}")
        # 3-5. 소스와 타겟을 합친 전체 문장 생성 (학습용 전체 텍스트)
        examples = [s + t for s, t in zip(sources, targets)]
        # 3-6. 토큰화 진행
        sources_tokenized = self._tokenize_fn(sources, tokenizer)  # source
        examples_tokenized = self._tokenize_fn(examples, tokenizer)  # source + target
        input_ids = examples_tokenized["input_ids"]
        labels = copy.deepcopy(input_ids) # 학습을 위한 정답지(Label) 생성
        # 3-7. 중요: Label Masking (질문 부분의 Loss 계산 제외)
        # 모델은 '질문'을 보고 '답변'을 생성해야 하므로, 질문 부분의 라벨은 -100으로 채워 학습에서 제
        for label, source_len in zip(labels, sources_tokenized["input_ids_lens"]):
            label[:source_len] = -100
        data_dict = dict(input_ids=input_ids, labels=labels)
        self.input_ids = data_dict["input_ids"]
        self.labels = data_dict["labels"]
        logging.warning("Loading data done!!: %d"%(len(self.labels)))
    def _tokenize_fn(self, strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
        """텍스트 리스트를 받아 토큰화하고 정보(ID, 길이 등)를 반환하는 보조 함수"""
        tokenized_list = [
            tokenizer(
                text,
                return_tensors="pt",
                padding="longest",
                max_length=tokenizer.model_max_length,
                truncation=True,
            )
            for text in strings
        ]
        # 실제 토큰 ID만 추출
        input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
        # 패딩을 제외한 실제 토큰의 길이 계산
        input_ids_lens = labels_lens = [
            tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item() for tokenized in tokenized_list
        ]
        return dict(
            input_ids=input_ids,
            labels=labels,
            input_ids_lens=input_ids_lens,
            labels_lens=labels_lens,
        )
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [ ]:
# 4. SFT 학습용 가변 길이 데이터 패딩 및 배치 처리기 (Data Collator)
@dataclass
class DataCollatorForSupervisedDataset(object):
    """
    SFT(Supervised Fine-Tuning) 학습 시, 가변 길이의 샘플들을
    하나의 배치로 묶고 패딩을 수행하는 클래스입니다.
    """
    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        # 4-1. 개별 샘플들(instances)로부터 input_ids와 labels를 추출하여 튜플로 만들기
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        # 4-2. input_ids 패딩: 문장들의 길이를 배치 내 최장 길이에 맞추기
        # 빈 공간은 토크나이저의 pad_token_id(보통 0 또는 3 등)로 채웁니다.
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        # 3. labels 패딩: 정답지(labels)도 같은 길이를 갖도록 패딩.
        # 중요: 패딩된 부분은 학습(Loss 계산)에 반영되지 않도록 -100으로 채
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value= -100)
        # 4. 최종 결과 반환
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

### 3-2. 학습실행

In [ ]:
# 5, SFT_dataset 클래스를 사용해 훈련셋을 만들고 data collator 인스턴스 생성
train_dataset = SFT_dataset(data_path_1_SFT=data_path_1_SFT, tokenizer=tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)
print('input : %s'%train_dataset.input_ids[0])
print('output: %s'%train_dataset.labels[0])

In [ ]:
# 6. Training arguments를 사용해 trainer 인스턴스 생성
output_path = os.path.join(project_path,'models')
training_args = transformers.TrainingArguments(
    output_dir=output_path,
    # overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=5,
    prediction_loss_only=True,
    fp16 = True
    )
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

In [ ]:
# 7. SFT 훈련 Test
trainer.train()
model_path = os.path.join(output_path,'output_1_SFT')
model.save_pretrained(model_path)

### 3-3. Generator 생성

In [ ]:
generator = transformers.pipeline('text-generation', model=model_path, tokenizer=tokenizer)

generation_args = dict(
    num_beams=4,
    repetition_penalty=2.0,
    no_repeat_ngram_size=4,
    eos_token_id=375, # \n
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    early_stopping=True
)

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = ['불고기용 고기 한우에요?',
               '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
               '시카고 오헤어 국제공항은 어디에 있어?',
               '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt' : tmp}) for tmp in list_prompt]

list_result = generator(list_prompt, **generation_args)
for prompt, result in zip(list_prompt, list_result):
    print()
    print((result[0]['generated_text']))

## 4.RM (Reward Model)

### 4-1. RM 설계

In [ ]:
# 메모리 관리를 위한 캐시 비우기
torch.cuda.empty_cache()

In [ ]:
# 1. 필요란 라이브러리 불러오기
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.strategies import NaiveStrategy
from chatgpt.trainer.rm import RewardModelTrainer
from transformers.models.gpt2.configuration_gpt2 import GPT2Config
from transformers.models.gpt2.modeling_gpt2 import GPT2Model
import torch.nn as nn
import random

In [ ]:
# 2. Reward model 설계
class GPTRM_custom(RewardModel):
    def __init__(self,
                 pretrained: Optional[str] = None,		# 사전 학습된 모델 경로 또는 이름
                 config: Optional[GPT2Config] = None, 	# 모델 설정 객체
                 checkpoint: bool = False,			# 그래디언트 체크포인팅 사용 여부 (메모리 절약)
                 lora_rank: int = 0,				# LoRA 적용 시 Rank 값 (0이면 미적용)
                 lora_train_bias: str = 'none',			# LoRA 학습 시 Bias 처리 방식
                 tokenizer=None) -> None:			# 토크나이저 객체
        # 1. 기반 모델(Backbone) 로드: GPT2Model을 베이스로 사용
        if pretrained is not None:
            # 허깅페이스에서 사전 학습된 GPT2 가중치를 불러오기
            model = GPT2Model.from_pretrained(pretrained)
            # 사용자의 토크나이저 크기에 맞춰 임베딩 레이어 크기를 재조정(한국어 토큰 대응)
            model.resize_token_embeddings(len(tokenizer))
        elif config is not None:
            # 가중치 없이 설정값만으로 모델 구조 생성
            model = GPT2Model(config)
        else:
            # 기본 GPT2 설정으로 모델을 생성
            model = GPT2Model(GPT2Config())
        # 2. 메모리 최적화 설정
        if checkpoint:
            # 학습 중 메모리 소비를 줄이기 위해 그래디언트 체크포인팅을 활성화
            model.gradient_checkpointing_enable()
        # 3. 보상 점수 출력층(Value Head) 정의
        # 모델의 마지막 은닉 상태(n_embd 차원)를 입력받아
        # 단 하나의 숫자(스칼라 값, 보상 점수)로 변환하는 선형 레이어입니다.
        value_head = nn.Linear(model.config.n_embd, 1)
        # 4. 부모 클래스(RewardModel) 초기화
        # 불러온 모델, 출력층, LoRA 설정 등을 부모 클래스에 전달
        super().__init__(model, value_head, lora_rank, lora_train_bias)
        if pretrained is not None:
            self.model = model
            self.pretrained = pretrained

    def save_pretrained(self, dir):
        """학습된 리워드 모델을 저장하는 함수"""
        if self.pretrained is not None:
            self.model.save_pretrained(dir)

In [ ]:
# 3. 사용할 모델과 토크나이저 로드
# 3-1. 기본 백본(Backbone) 모델 로드
# SKT에서 공개한 KoGPT2 가중치를 불러옵니다.
# (참고: 아래에서 GPTRM_custom 내부적으로 다시 로드하므로, 이 줄은 가중치 다운로드 확인용이나 참조용으로 쓰)
model = AutoModelForCausalLM.from_pretrained('skt/kogpt2-base-v2')
# 3-2. 토크나이저 설정
# KoGPT2는 BOS, EOS 등의 구분이 명확하지 않은 경우가 많아 특수 토큰을 </s>로 통일
# from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2", model_max_length=512,
                                                    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
                                                    pad_token='<pad>', mask_token='<mask>')

# tokenizer = AutoTokenizer.from_pretrained(
#         'skt/kogpt2-base-v2', bos_token='</s>', # 문장의 시작을 알리는 토큰
#         eos_token='</s>', # 문장의 끝을 알리는 토큰
#         unk_token='</s>', # 모르는 단어를 처리하는 토큰
#         pad_token='</s>', # 문장 길이를 맞추기 위한 패딩 토큰
#         padding_side="right", # 패딩을 문장의 오른쪽에 채움 (Causal LM의 일반적인 방식)
#         model_max_length=512, # 모델이 처리할 수 있는 최대 문장 길이 설정
# )
# 3-3. 분산 학습 전략(Strategy) 설정 및 커스텀 리워드 모델 초기화
# NaiveStrategy는 단일 GPU나 기본적인 학습 환경에서 사용되는 초기화 전략
with NaiveStrategy().model_init_context():
        # 앞서 정의한 GPTRM_custom 클래스를 인스턴스화
        # pretrained: 기반이 될 모델 이름
        # lora_rank=0: LoRA를 사용하지 않고 전체 가중치를 학습
        # .cuda(): 생성된 모델을 GPU 메모리로 이동시켜 연산 속도를 높임
        model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', lora_rank=0, tokenizer=tokenizer).cuda()

In [ ]:
# 4. RM을 훈련시킬 때 사용할 ranking dataset 만들기
# 4-1. 원본 데이터 로드
dataset_path = os.path.join(project_path, 'data_kochatgpt/kochatgpt_2_RM.jsonl')
with open(dataset_path, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
total_data_ranking2chosen = []
# 4-2.  데이터를 순회하며 답변 간의 우열을 가려 '쌍(Pair)'으로 만듦
for tmp in list_data_dict:
    one_data_ranking2chosen = []
    # --- 경우의 수 1: 답변 0과 답변 1 비교 ---
    data = {}
    data['prompt'] = tmp['prompt']
    # ranking 숫자가 작을수록 더 높은 순위(우수한 답변)라고 가정함
    if tmp['ranking'][0] < tmp['ranking'][1]:
        data['chosen'] = tmp['completion_0']	# 더 좋은 답변
        data['rejected'] = tmp['completion_1']	# 더 나쁜 답변
    else:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)
    # --- 경우의 수 2: 답변 0과 답변 2 비교 ---
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][0] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_0']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_0']
    one_data_ranking2chosen.append(data)
    # --- 경우의 수 3: 답변 1과 답변 2 비교 ---
    data = {}
    data['prompt'] = tmp['prompt']
    if tmp['ranking'][1] < tmp['ranking'][2]:
        data['chosen'] = tmp['completion_1']
        data['rejected'] = tmp['completion_2']
    else:
        data['chosen'] = tmp['completion_2']
        data['rejected'] = tmp['completion_1']
    one_data_ranking2chosen.append(data)
    # 4-3 한 질문에서 나온 3쌍의 비교 데이터를 전체 리스트에 합침
    total_data_ranking2chosen.extend(one_data_ranking2chosen)
# 4-3 결과 확인
print('before data num: %d'%(len(list_data_dict)))	# 원본 질문 개수
print('after  data num: %d'%(len(total_data_ranking2chosen)))	# 생성된 비교 쌍(Pair) 총 개수
print('data example: \n%s'%total_data_ranking2chosen[45])

※ 데이터 구성 원리
- 원본 소스: 동일한 질문에 대해 3개의 모델(ChatGPT, Davinci, Ada)이 답변 생성.
- 품질 기준: ChatGPT(Good) > Davinci(Bad) > Ada(Worst)로 순위가 이미 매겨져 있음.
- 저장 방식: completion_0, 1, 2에 모델 답변을 무작위로 할당 (예: 0번에 Ada, 1번에 ChatGPT가 들어갈 수 있음).
- 정답지: ranking 키에 각 답변의 실제 순위를 기록 (예: ranking: [2, 0, 1]이라면 1번 답변이 가장 우수하다는 뜻).
  
※ RM의 loss function

```
class PairWiseLoss(nn.Module):

    def forward(self, chosen_reward: torch.Tensor, reject_reward: torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(chosen_reward - reject_reward)
        log_probs = torch.log(probs)
        loss = -log_probs.mean()
        return loss
```


In [ ]:
# 5. 완성한 ranking dataset을 shuffle한 후 훈련셋 만들기
# 빠른 학습을 위해 전체 데이터중 일부만 학습
import random
# 5-1. 실험의 재현성을 위해 랜덤 시드 고정
# 같은 시드 번호를 사용하면 코드를 다시 실행해도 똑같은 순서로 데이터가 섞
random.seed(230319)
#5-2. 생성된 전체 비교 데이터(Pairwise)를 무작위로 섞음
# 데이터의 특정 패턴(예: 특정 질문이 앞부분에 몰려 있는 현상)으로 인해 학습이 편향되는 것을 방지
random.shuffle(total_data_ranking2chosen)
# 셔플 결과 확인을 위해 특정 인덱스의 데이터 출력
print(total_data_ranking2chosen[45])

In [ ]:
# 5-3 데이터 슬라이싱을 통한 훈련셋과 검증셋 분할
train_data = total_data_ranking2chosen[:1000]
eval_data = total_data_ranking2chosen[1000:1200]
print(len(train_data))
print(len(eval_data))
# 5-4텍스트 데이터를 모델이 이해할 수 있는 텐서(Tensor) 형태로 변환
train_dataset = RewardDataset(train_data, tokenizer, 512)
eval_dataset = RewardDataset(eval_data, tokenizer, 512)

In [ ]:
# 6. 데이터 셋 확인
idx = 1
print('#'*70)
print('## prompt ##')
print(train_data[idx]['prompt'])
print('#'*70)
print('## chosen ##')
print(train_data[idx]['chosen'])
print('#'*70)
print('## rejected ##')
print(train_data[idx]['rejected'])


### 4-2. RM학습

In [ ]:
trainer = RewardModelTrainer(model=model,
                             strategy=NaiveStrategy(),
                             optim=torch.optim.Adam(model.parameters(), lr=5e-5),
                             train_dataset=train_dataset,
                             eval_dataset=eval_dataset,
                             batch_size=4,
                             max_epochs=1)

In [ ]:
model_path = os.path.join(output_path,'output_2_RM')
trainer.fit(use_lora=0)
model.save_pretrained(model_path)

### 4-3. RM 학습 결과 확인

In [ ]:
def inference_RM(input_text):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').cuda()
    output = model(input_ids)
    output_reward = output.cpu().detach().numpy()[0]
    print('input: %s\nreward score: %.1f'%(input_text, output_reward))
    return output_reward
input_text = '인공지능은 똥멍청이 입니다'
output_reward = inference_RM(input_text=input_text)

In [ ]:
input_text = '인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다.'
output_reward = inference_RM(input_text=input_text)

In [ ]:
input_text = "인공지능(AI)은 컴퓨터에서 음성 및 작성된 언어를 보고 이해하고 번역하고 데이터를 분석하고 추천하는 기능을 포함하여 다양한 고급 기능을 수행할 수 있는 일련의 기술입니다. AI는 현대적인 컴퓨팅 혁신에서 중추적인 역할을 하며 개인과 비즈니스의 가치를 창출합니다. 예를 들어 광학 문자 인식(OCR)은 AI를 사용해 이미지 및 문서에서 텍스트 및 데이터를 추출하고, 구조화되지 않은 콘텐츠를 비즈니스에 바로 사용할 수 있게 만들고, 유용한 정보를 창출합니다."
output_reward = inference_RM(input_text=input_text)

In [ ]:
input_text = "인공지능은 일반적으로 인간의 지능이 필요하거나 인간이 분석할 수 있는 것보다 규모가 큰 데이터를 포함하는 방식으로 추론, 학습 및 행동할 수 있는 컴퓨터 및 기계를 구축하는 것과 관련된 과학 분야입니다. AI는 컴퓨터 공학, 데이터 분석 및 통계, 하드웨어 및 소프트웨어 엔지니어링, 언어학, 신경 과학은 물론 철학과 심리학을 포함하여 여러 학문을 포괄하는 광범위한 분야입니다. 비즈니스의 운영 수준에서 AI는 주로 머신러닝과 딥 러닝을 기반으로 하는 기술 모음으로, 데이터 분석, 예상 및 예측, 객체 분류, 자연어 처리, 추천, 지능형 데이터 가져오기 등을 수행할 수 있습니다."
output_reward = inference_RM(input_text=input_text)

### 📊 KoChatGPT 리워드 모델(RM) 테스트 결과 분석 보고

| 분석 항목 | 분석 내용 (Detail) | 주요 원인 및 의미 (Key Point) |
| :--- | :--- | :--- |
| **품질 비례 점수 상승** | **부분적 일치.** T1(-2.5) → T3(-1.7)까지는 품질에 따라 점수가 상승했으나, 가장 정보량이 많은 T4(-2.2)에서 오히려 하락함. | **길이 편향(Length Bias):** 모델이 특정 길이 이상의 문장을 학습 데이터의 분포를 벗어난 것으로 판단했거나 학습 부족 가능성. |
| **점수 값의 적절성** | **상대적 순위는 적절.** 비하 표현(T1)에 가장 낮은 점수를 준 것은 긍정적이나, 전문성(T4) 판별력은 아직 낮음. | **문체 의존성:** 정교한 논리 구조보다는 '전형적인 설명투'의 문장에 더 높은 점수를 주는 경향이 있음. |
| **음수(-) 점수의 의미** | **상대적 선호도 지표.** 0을 기준으로 한 절대적 가치가 아니라, 학습 데이터의 '평균' 대비 현재 입력의 위치를 의미함. | **Baseline 비교:** 강화학습(PPO) 단계에서 다른 답변들보다 '상대적으로' 얼마나 더 선택되어야 하는지를 결정하는 척도. |
| **음수 출력 허용 방법** | **활성화 함수 제거.** 마지막 `nn.Linear` 레이어 뒤에 `Sigmoid`나 `ReLU` 등을 사용하지 않음. | **출력 범위 확보:** 선형 레이어만 사용하여 $(-\infty, \infty)$ 범위의 실수를 출력하도록 설계하여 자유로운 비교 가능. |
| **스칼라(Scalar)의 중요성** | **단일 비교 및 최적화.** 여러 답변 중 무엇이 더 좋은지 일직선상에서 비교하기 위해 하나의 숫자로 압축해야 함. | **PPO 알고리즘 연동:** 강화학습은 '단일 보상값'을 기준으로 가중치를 업데이트하므로, 다차원 벡터가 아닌 1차원 스칼라가 필수적임. |

---

### 💡 종합 의견 및 향후 과제
현재 RM은 기본적인 **부정적 텍스트 필터링** 능력은 갖추었으나, **텍스트의 복잡도나 전문성**을 완벽히 변별하기에는 데이터셋의 다양성 보강이 필요해 보입니다. 향후 PPO 학습 단계에서 이 리워드 스코어가 모델의 정책(Policy)을 결정하는 핵심 보상값으로 작용하게 됩니다.

## 5.PPO-Proximal Policy Optimization

### 5-1. PPO모델 구현

In [ ]:
# 1. 라이브러리 불러오기
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer
from copy import deepcopy

In [ ]:
# 2. 사용할 모델 설정
actor_model = os.path.join(project_path, 'models/output_1_SFT/checkpoint-1500')
critic_model = os.path.join(project_path, 'models/output_2_RM')
with NaiveStrategy().model_init_context():
    actor = GPTActor(pretrained=actor_model, lora_rank=0).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained=critic_model, lora_rank=0).to(torch.cuda.current_device())
    tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2", model_max_length=512,
                                                    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
                                                    pad_token='<pad>', mask_token='<mask>')
    initial_model = deepcopy(actor)
    reward_model = RewardModel(deepcopy(critic.model), deepcopy(critic.value_head)).to(torch.cuda.current_device())

In [ ]:
# 2-1. 옵티마이저와 모델 준비
actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

In [ ]:
(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

### 5-2. PPOTrainer 클래스를 설계

In [ ]:
# 3. PPO 학습에 쓸 데이터를 불러와 토크나이징
tokenizer_json = os.path.join(project_path, 'data_kochatgpt/kochatgpt_3_PPO.jsonl')
with open(tokenizer_json, "r", encoding='utf-8-sig') as json_file:
    list_data_dict = json.load(json_file)
    list_prompt = [tmp['prompt'] for tmp in list_data_dict]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

In [ ]:
print(tokenize_fn('It takes something more than intelligence to act intelligently.'))

In [ ]:
test_text = '불고기용 고기 한우에요?'
tokenized = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokenized)

print(f"테스트 결과: {decoded}")
print(f"정상 토큰 ID 예시: {tokenized[:5]}") # 9000번 이상의 큰 숫자가 나와야 성공!

In [ ]:
len(list_prompt)

In [ ]:
# 4. PPOTrainer 클래스를 설계하여 학습
trainer = PPOTrainer(NaiveStrategy(),
                     actor,
                     critic,
                     reward_model,
                     initial_model,
                     actor_optim,
                     critic_optim,
                     max_epochs=1,
                     train_batch_size=8,
                     tokenizer=tokenize_fn,
                     max_length=128,
                     do_sample=True,
                     temperature=1.0,
                     top_k=50,
                     pad_token_id=tokenizer.pad_token_id,
                     eos_token_id=tokenizer.eos_token_id)

In [ ]:
trainer.fit(list_prompt,
            num_episodes=10,
            max_timesteps=3,
            update_timesteps=3)

actor.model.save_pretrained('models/output_3_PPO')

### 5-3. RLHF가 적용된 koGPT-2의 생성능력을 확인

In [ ]:
def generation(input_text, model):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(
        torch.cuda.current_device())
    outputs = model.generate(input_ids,
                             max_length=250,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             num_return_sequences=1)
    output = tokenizer.batch_decode(outputs[0], skip_special_tokens=True)
    print()
    print(output)
    return output

PROMPT_DICT = {
    "prompt_input": (
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

list_prompt = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?']

list_prompt = [PROMPT_DICT['prompt_input'].format_map({'prompt': tmp}) for tmp in list_prompt]

for input_text in list_prompt:
    print(list_prompt[0])
    output = generation(input_text, actor)